In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import gseapy as gp
import squidpy as sq
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colorbar as colorbar
import matplotlib.colors as mcolors

from anndata import AnnData
from typing import Tuple, Literal
from scipy.stats import pearsonr
from scipy.ndimage import gaussian_filter
from skimage.draw import polygon
from skimage.measure import find_contours

In [ ]:
for mod in [np, pd, sc, gp, sq]:
    print(mod, mod.__version__)

In [ ]:
def get_genesets(
    adata: AnnData,
    df_hcd_all: pd.DataFrame,
    celltype_signature: str,
    direction: str,
    threshold_pval: float,
    threshold_lfc: float, 
    threshold_frac_shared_genes: float,
) -> Tuple[dict[str, list[str]], pd.DataFrame]:

    # construct signature gene set
    select = ((df_hcd_all.adj_p_value <= threshold_pval) & (df_hcd_all.celltype == celltype_signature))
    if direction == "upregulated":
        select = select & (df_hcd_all.log_fc >= threshold_lfc)
    elif direction == "downregulated":
        select = select & (df_hcd_all.log_fc <= threshold_lfc)
    elif direction == "both":
        select = select & (df_hcd_all.log_fc.abs() >= threshold_lfc)         
    
    df_hcd_ct = df_hcd_all.loc[select]
    gene_set_dict = {}
    gene_set_df = pd.DataFrame()
    for cytokine_i, cytokine in enumerate(df_hcd_ct.cytokine.unique()):
        gene_set = df_hcd_ct.loc[df_hcd_ct.cytokine == cytokine].gene.values
        gene_set_shared = np.intersect1d(gene_set, adata.var_names)
        frac_shared = len(gene_set_shared) / len(gene_set)
        
        row_idx = gene_set_df.shape[0]
        gene_set_df.loc[row_idx, "cytokine"] = cytokine
        gene_set_df.loc[row_idx, "num_genes_signature"] = len(gene_set)
        gene_set_df.loc[row_idx, "num_shared_genes_signature"] = len(gene_set_shared)
        gene_set_df.loc[row_idx, "frac_shared_genes_signature"] = frac_shared
        gene_set_df.loc[row_idx, "geneset"] = [[gene_set]]
        gene_set_df.loc[row_idx, "geneset_shared"] = [[gene_set_shared]]
        if len(gene_set_shared) > 0 and frac_shared >= threshold_frac_shared_genes:
            gene_set_dict[cytokine] = gene_set_shared
        # else:
            # print(f"No shared genes found for: {cytokine}")

    return gene_set_dict, gene_set_df

In [ ]:
def get_genesets(
    adata: AnnData,
    df_hcd_all: pd.DataFrame,
    celltype_signature: str,
    direction: str,
    threshold_pval: float,
    threshold_lfc: float,
    threshold_frac_shared_genes: float,
    min_size: int = None,
    max_size: int = None,
) -> Tuple[dict[str, list[str]], pd.DataFrame]:

    # construct signature gene set
    select = ((df_hcd_all.adj_p_value <= threshold_pval) & (df_hcd_all.celltype == celltype_signature))
    if direction == "upregulated":
        select = select & (df_hcd_all.log_fc >= threshold_lfc)
    elif direction == "downregulated":
        select = select & (df_hcd_all.log_fc <= threshold_lfc)
    elif direction == "both":
        select = select & (df_hcd_all.log_fc.abs() >= threshold_lfc)         
    
    df_hcd_ct = df_hcd_all.loc[select]
    gene_set_dict = {}
    gene_set_df = pd.DataFrame()
    for cytokine_i, cytokine in enumerate(df_hcd_ct.cytokine.unique()):
        gene_set = df_hcd_ct.loc[df_hcd_ct.cytokine == cytokine].gene.values
        gene_set_shared = np.intersect1d(gene_set, adata.var_names)
        frac_shared = len(gene_set_shared) / len(gene_set)
        
        row_idx = gene_set_df.shape[0]
        gene_set_df.loc[row_idx, "cytokine"] = cytokine
        gene_set_df.loc[row_idx, "num_genes_signature"] = len(gene_set)
        gene_set_df.loc[row_idx, "num_shared_genes_signature"] = len(gene_set_shared)
        gene_set_df.loc[row_idx, "frac_shared_genes_signature"] = frac_shared
        gene_set_df.loc[row_idx, "geneset"] = [[gene_set]]
        gene_set_df.loc[row_idx, "geneset_shared"] = [[gene_set_shared]]
        if (
            (len(gene_set_shared) > 0) 
            and (frac_shared >= threshold_frac_shared_genes) 
            and (len(gene_set_shared) >= min_size)
            and len(gene_set_shared) <= max_size
        ):
            gene_set_dict[cytokine] = gene_set_shared

    return gene_set_dict, gene_set_df


def compute_s2n(adata: AnnData, contrast_column: str, condition_1: str, condition_2: str) -> (pd.DataFrame, pd.DataFrame):
    """
    Compute the signal-to-noise ratio (S2N) for each gene between two conditions in an AnnData object.

    Parameters:
    - adata: AnnData object with gene expression data.
    - contrast_column: Key in `adata.obs` indicating the condition labels (e.g. "disease_state").
    - condition_1: Name of the first condition (e.g., "flare").
    - condition_2: Name of the second condition (e.g., "healthy").
    
    Returns:
    - s2n_scores: pandas Series of S2N values indexed by gene names.
    """
    # Select cells for each condition
    group1 = adata[adata.obs[contrast_column] == condition_1]
    group2 = adata[adata.obs[contrast_column] == condition_2]

    # number of cells per condition
    num_cells_1 = group1.shape[0]
    num_cells_2 = group2.shape[0]
    
    # Get expression matrices
    X1 = group1.X.toarray() if hasattr(group1.X, "toarray") else group1.X
    X2 = group2.X.toarray() if hasattr(group2.X, "toarray") else group2.X

    # Compute mean and std per gene
    mu1 = np.mean(X1, axis=0)
    mu2 = np.mean(X2, axis=0)
    sigma1 = np.std(X1, axis=0, ddof=1)
    sigma2 = np.std(X2, axis=0, ddof=1)

    # Compute S2N
    s2n = (mu1 - mu2) / (sigma1 + sigma2 + 1e-8)  # epsilon to avoid division by zero

    num_cells = pd.DataFrame(index=[f"{condition_1}_vs_{condition_2}"], columns=["num_cells_1", "num_cells_2"], data=[[num_cells_1, num_cells_2]])
    stats = pd.DataFrame(s2n, index=adata.var_names, columns=[f"{condition_1}_vs_{condition_2}"])
    
    return stats, num_cells

  
def compute_ranking_statistic(adata: AnnData, contrast_column: str, contrasts: list[Tuple[str, str]]) -> (pd.DataFrame, pd.DataFrame):
    rnk_stats, num_cells = [], []
    for condition in contrasts:
        _rnk_stats, _num_cells = compute_s2n(
            adata,
            contrast_column=contrast_column,
            condition_1=condition[0],
            condition_2=condition[1]
        )
        rnk_stats.append(_rnk_stats)
        num_cells.append(_num_cells)
    return pd.concat(rnk_stats, axis=1), pd.concat(num_cells, axis=0)

    
def run_enrichment_test(
    adata: AnnData,
    df_hcd_all: pd.DataFrame,
    celltype: Tuple[str, str] = ("B cell", "B"),
    direction: str = "upregulated",
    threshold_pval: float = 0.01,
    threshold_lfc: float = 1.,
    threshold_expression: float = 0,
    threshold_frac_shared_genes: float = 0,
    contrast_column: str = "disease_state",
    celltype_column: str = "disease_state",
    contrasts: Tuple[str, str] | list[Tuple[str, str]] = None,
    min_size: int = 10,
    max_size: int = 1000,
    permutation_num: int = 1000,
    weight: float = 1.,
    max_obs_per_condition: int | None = None, # None = all
    seed: int = 2025,
    verbose: bool = True,
    threads: int = 6,
) -> pd.DataFrame:

    if not isinstance(contrasts, list):
        assert isinstance(contrasts, Tuple)
        contrasts = [contrasts]
        
    celltype_adata = celltype[0]
    celltype_signature = celltype[1]

    # filter for cell type
    adata = adata[adata.obs[celltype_column] == celltype_adata]
    
    # filter based on gene expression
    adata = adata[:, adata.X.mean(axis=0) >= threshold_expression]

    # get genesets
    gene_set_dict, gene_set_df = get_genesets(
        adata=adata,
        df_hcd_all=df_hcd_all,
        celltype_signature=celltype_signature,
        direction=direction,
        threshold_pval=threshold_pval,
        threshold_lfc=threshold_lfc,
        threshold_frac_shared_genes=threshold_frac_shared_genes,
        min_size=min_size,
        max_size=max_size,
    )

    # return adata, gene_set_dict

    if len(gene_set_dict) == 0:
        print("No gene sets passed filtering criteria.")
        return None
    
    # compute ranking stat
    rnk_stats, num_cells_per_condition = compute_ranking_statistic(adata, contrast_column=contrast_column, contrasts=contrasts)
    results = []
    for contrast_name in rnk_stats.columns:
        
        # format stat so that it can be processed be gseapy
        
        rnk = rnk_stats.loc[:, contrast_name]\
            .replace([np.inf, -np.inf], np.nan)\
            .dropna()\
            .sort_values(ascending=False)\
            .to_frame()\
            .rename({contrast_name: 1}, axis=1)\
            .rename_axis("0")
        
        # run enrichment
        gp_res = gp.prerank(
            rnk=rnk,
            gene_sets=gene_set_dict,
            min_size=min_size,
            max_size=max_size,
            permutation_num=permutation_num,
            weight=weight,
            outdir=None,
            seed=seed,
            verbose=verbose,
            threads=threads,
        )
        _res = gp_res.res2d
        _res.loc[:, "contrast"] = contrast_name
        _res.loc[:, "num_cells_1"] = num_cells_per_condition.loc[contrast_name, "num_cells_1"]
        _res.loc[:, "num_cells_2"] = num_cells_per_condition.loc[contrast_name, "num_cells_2"]
        _res.loc[:, "percent_duplicate_ranking_stats"] = (rnk.duplicated(keep="first").sum() / rnk.shape[0]) * 100
        
        results.append(_res)

    # combine results and save hyperparams
    results = pd.concat(results, axis=0, ignore_index=True)
    results.loc[:, "celltype_adata"] = celltype_adata
    results.loc[:, "celltype_signature"] = celltype_signature
    results.loc[:, "celltype_combo"] = f"{celltype_adata} ({celltype_signature})"
    results.loc[:, "direction"] = direction
    results.loc[:, "threshold_pval"] = threshold_pval
    results.loc[:, "threshold_lfc"] = threshold_lfc
    results.loc[:, "threshold_expression"] = threshold_expression
    results.loc[:, "min_size"] = min_size
    results.loc[:, "max_size"] = max_size
    results.loc[:, "permutation_num"] = permutation_num
    results.loc[:, "weight"] = weight
    results.loc[:, "seed"] = seed
    results.loc[:, "threads"] = threads

    results.rename({"Term": "cytokine"}, inplace=True, axis=1)
    results = pd.merge(results, gene_set_df, on="cytokine")
    
    return results

### load and prepare human cytokine dictionary

In [ ]:
df_hcd_all = pd.read_csv("/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_all_with_pbs_DE_count.csv", index_col=0)
questionable_genes = set(pd.read_csv("/Users/soeren.becker/repos/HumanCytokineDict/notebooks/01_overview/questionable_genes.csv")['gene'])
df_hcd_all = df_hcd_all.loc[
    (
        (df_hcd_all['pbs_well'] > 3) 
        & (~df_hcd_all['gene'].isin(questionable_genes))
        & (df_hcd_all['adj_p_value'] < 0.05)
    )
]

In [ ]:
df_hcd_all.celltype.unique()

### load and prepare dataset

In [ ]:
adata = sc.read_h5ad("/Users/soeren.becker/repos/HumanCytokineDict/data/use_case_datasets/07_spatial/NSCLC_3D.h5ad")
adata = adata[adata.obs.niches_3D != "Excluded"]
# adata.layers["raw"] = adata.X
# adata.X = adata.layers["SCT"] # normalized data
# adata = adata[adata.obs.niches_3D != "Excluded"]
# adata.obs.celltypes.unique().tolist()

adata.X.min(), adata.X.max()

In [ ]:
sc.pp.normalize_total(adata, target_sum=10_000)
sc.pp.log1p(adata)

In [ ]:
adata.X.min(), adata.X.max()

In [ ]:
# define celltype-signature combinations
celltype_combos = [
    ("Cytotoxic T cells", "CD8_T_cell"),
    ("Monocytes", "CD14_Mono"),
    ("Monocytes", "CD16_Mono"),
    ("Regulatory T cells", "Treg"),
    ("B cells", "B cell"),
    ("Dendritic cells", "cDC"),
]

In [ ]:
all_niches = []
for niche in adata.obs.niches_3D.unique():
    if niche in ["Tumor core", "Tumor surface"]:
        continue
    all_niches.append(niche.replace(" ", "_"))
        
    adata.obs.loc[:, "tumor_vs_niche"] = adata.obs.loc[:, "niches_3D"].apply(
        lambda x: "tumor" if x in ["Tumor core", "Tumor surface"] else x.replace(" ", "_")
    )

In [ ]:
adata.obs

In [ ]:
all_contrasts = [("tumor", niche) for niche in all_niches]
all_contrasts

In [ ]:
os.makedirs("./results_finalized_DEGs", exist_ok=True)

In [ ]:
all_results = []

# for threshold_expression in [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]:
for threshold_expression in [0.05, 0.06, 0.07, 0.08, 0.09, 0.1]:    
    for threshold_lfc in [0.8, 0.9, 1, 1.1, 1.2]:
        shared_kwargs = {
            "threshold_pval": 0.05,
            "threshold_lfc": threshold_lfc,
            "threshold_expression": threshold_expression,
            "threshold_frac_shared_genes": 0.1,
            "direction": "upregulated",
            "min_size": 8,
            "max_size": 1_000,
            "permutation_num": 20_000,
            "verbose": True,
            "adata": adata,
            "df_hcd_all": df_hcd_all,
            "seed": 2025,
        }
        
        for celltype_combo_k, celltype_combo in enumerate(celltype_combos):
            print(threshold_expression, threshold_lfc, celltype_combo)
            ### selected contrasts ###
            results = run_enrichment_test(
                celltype=celltype_combo,
                contrast_column="tumor_vs_niche",
                celltype_column="celltypes",
                contrasts=all_contrasts,
                **shared_kwargs,
            )
            if results is None: 
                # no gene set passed filtering
                continue
            
            all_results.append(results)
        if len(all_results) > 0:
            results = pd.concat(all_results, axis=0, ignore_index=True)
            results.to_csv("./results_finalized_DEGs/results_enrichment_tumor_vs_non-tumor_renormalized.csv")

In [ ]:
results = pd.read_csv("./results_finalized_DEGs/results_enrichment_tumor_vs_non-tumor_renormalized.csv", index_col=0)

In [ ]:
results.loc[results.percent_duplicate_ranking_stats < 0.5].shape[0] / results.shape[0]

In [ ]:
results.percent_duplicate_ranking_stats.max()

In [ ]:
results = results.loc[results.percent_duplicate_ranking_stats < 0.5]

In [ ]:
from tqdm.auto import tqdm
import warnings

def _check_robustness_fractions(
    df_pivot, 
    threshold_qval=0.1, # adjusted p value
    # threshold_valid=0.8, # fraction of results required to even consider this condition. I.e. if the test only ran for one set of thresholds, then it is not very robust.
    threshold_valid=0.5,
    threshold_below_alpha=0.75, # fraction of results that need to be significant
):
    n_total = np.prod(df_pivot.shape)
    n_valid = n_total - df_pivot.isna().sum().sum()
    n_below_alpha = (df_pivot < threshold_qval).sum().sum() # number of results below pval threshold, i.e., number of significant results
    frac_valid_results = (n_valid / n_total)
    frac_pval_below_alpha = (n_below_alpha / n_valid) # fraction of significant results relative to valid results
    is_robust = (frac_pval_below_alpha > threshold_below_alpha) & (frac_valid_results > threshold_valid)
    return frac_valid_results, frac_pval_below_alpha, is_robust

def check_robustness(
    all_results,
    THRESHOLD_QVAL = 0.1,
    THRESHOLD_VALID = 0.8,
    THRESHOLD_BELOW_ALPHA = 0.9,
):
    
    all_thresholds_expression = all_results.threshold_expression.sort_values(ascending = False).unique()
    all_thresholds_lfc = sorted(all_results.threshold_lfc.unique())
    
    df = pd.DataFrame(index=all_thresholds_expression, columns=all_thresholds_lfc)
    df.index.rename("threshold_expression", inplace=True)
    df.columns.rename("threshold_lfc", inplace=True)
    
    robust_results = []
    
    for contrast in tqdm(all_results.contrast.unique()):
        for celltype_combo in all_results.celltype_combo.unique():
            results_ct = all_results.loc[(all_results.celltype_combo == celltype_combo) & (all_results.contrast == contrast)]
            for cytokine in results_ct.cytokine.unique():
                results_ct_cy = results_ct.loc[results_ct.cytokine == cytokine]
                df_pivot = results_ct_cy.pivot(index="threshold_expression", columns="threshold_lfc", values="FDR q-val")
                df_pivot_nes = results_ct_cy.pivot(index="threshold_expression", columns="threshold_lfc", values="NES")
                row_order = df_pivot.index.sort_values(ascending=False)
                with warnings.catch_warnings():
                    warnings.simplefilter(action='ignore', category=FutureWarning)
                    df_combined = pd.concat([df, df_pivot])
                    df_combined_nes = pd.concat([df, df_pivot_nes])
                df_merged = df_combined.combine_first(df_pivot)
                df_merged = df_merged.loc[~df_merged.index.duplicated()]
                df_pivot = df_merged.loc[all_thresholds_expression, all_thresholds_lfc].astype(float)

                df_merged_nes = df_combined_nes.combine_first(df_pivot_nes)
                df_merged_nes = df_merged_nes.loc[~df_merged_nes.index.duplicated()]
                df_pivot_nes = df_merged_nes.loc[all_thresholds_expression, all_thresholds_lfc].astype(float)
                
                # sns.heatmap(df_pivot, annot=True, vmin=0, vmax=0.05, cmap="Reds_r")
                # plt.title(f"{contrast} {celltype_combo} {cytokine}")
                # plt.show()
                
                frac_valid_results, frac_pval_below_alpha, is_robust = _check_robustness_fractions(
                    df_pivot, 
                    threshold_qval=THRESHOLD_QVAL,
                    threshold_valid=THRESHOLD_VALID,
                    threshold_below_alpha=THRESHOLD_BELOW_ALPHA,
                )
                
                if is_robust:

                    if (df_pivot_nes.min().min() < 0) and (df_pivot_nes.max().max() > 0):
                        sns.heatmap(df_pivot, annot=True, vmin=0, vmax=0.05, cmap="Reds_r")
                        plt.title(f"{contrast} {celltype_combo} {cytokine}")
                        plt.show()
    
                        sns.heatmap(df_pivot_nes, annot=True, fmt=".2f", center=0, cmap="RdBu_r")
                        plt.title(f"{contrast} {celltype_combo} {cytokine}")
                        plt.show()
                        
                    robust_results.append(
                        (
                            celltype_combo, contrast, cytokine,
                            frac_valid_results, frac_pval_below_alpha, is_robust,
                            results_ct_cy.NES.min(), results_ct_cy.NES.max(),
                            THRESHOLD_QVAL,
                            THRESHOLD_BELOW_ALPHA,
                        )
                    )
        
    robust_results = pd.DataFrame(robust_results).rename(
        {0: "celltype_combo", 1: "contrast", 2: "cytokine", 3: "frac_valid", 4: "frac_significant", 5:"is_robust", 6:"NES_min", 7:"NES_max", 8:"qval_threshold", 9: "threshold_frac_below_alpha"},
        axis=1
    )
    return robust_results

In [ ]:
# robust_results = check_robustness(results.loc[results.cytokine == "IFN-gamma"], THRESHOLD_QVAL=0.1, THRESHOLD_BELOW_ALPHA=0.1)
if False:
    robust_results_01 = check_robustness(results, THRESHOLD_QVAL=0.1, THRESHOLD_VALID=1/3, THRESHOLD_BELOW_ALPHA=0.9)
    robust_results_005 = check_robustness(results, THRESHOLD_QVAL=0.05, THRESHOLD_VALID=1/3, THRESHOLD_BELOW_ALPHA=0.9)
    robust_results_001 = check_robustness(results, THRESHOLD_QVAL=0.01, THRESHOLD_VALID=1/3, THRESHOLD_BELOW_ALPHA=0.9)

THRESHOLD_VALID = 0.5
THRESHOLD_BELOW_ALPHA = 2/3

if True:
    robust_results_01 = check_robustness(results, THRESHOLD_QVAL=0.1, THRESHOLD_VALID=THRESHOLD_VALID, THRESHOLD_BELOW_ALPHA=THRESHOLD_BELOW_ALPHA)
    robust_results_005 = check_robustness(results, THRESHOLD_QVAL=0.05, THRESHOLD_VALID=THRESHOLD_VALID, THRESHOLD_BELOW_ALPHA=THRESHOLD_BELOW_ALPHA)
    robust_results_001 = check_robustness(results, THRESHOLD_QVAL=0.01, THRESHOLD_VALID=THRESHOLD_VALID, THRESHOLD_BELOW_ALPHA=THRESHOLD_BELOW_ALPHA)


robust_results = pd.concat([robust_results_01, robust_results_005, robust_results_001])
robust_results = robust_results.groupby(["contrast", "celltype_combo", "cytokine"])["qval_threshold"].min().to_frame().reset_index()

In [ ]:
robust_results

In [ ]:
idcs = robust_results.set_index(["celltype_combo", "cytokine"]).index
_results = results.set_index(["celltype_combo", "cytokine"])
selected_results = _results.loc[_results.index.intersection(idcs)].reset_index()
selected_results = selected_results.groupby(["contrast", "celltype_combo", "cytokine"])["NES"].mean().to_frame().reset_index()

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
def replace_greek(x):
    if isinstance(x, (list, np.ndarray, pd.Index)):
        return [replace_greek(l) for l in x]
    if not isinstance(x, str):
        x = x.get_text()
    x = x.replace("alpha", "α")
    x = x.replace("beta", "β")
    x = x.replace("gamma", "γ")
    x = x.replace("lambda", "λ")
    x = x.replace("epsilon", "ε")    
    x = x.replace("omega", "ω")
    return x

def shorten_celltype_labels(cell_type):
    if isinstance(cell_type, (list, np.ndarray, pd.Index)):
        return [shorten_celltype_labels(_cell_type) for _cell_type in cell_type]    

    if not isinstance(cell_type, str):
        cell_type = cell_type.get_text()
    
    if 'Monocytes' in cell_type:
        simplified_label = cell_type.replace("_Mono", "").replace("Monocytes", "Mono.")
    else:
        simplified_label = cell_type.split("(")[0] 
        simplified_label = simplified_label.replace(" cells", "")
    return simplified_label

def extract_niche_name(labels):
    niche_names = []
    for label in labels:
        text = label.get_text() if hasattr(label, 'get_text') else label
        parts = text.split('vs_', 1)
        if len(parts) == 2:
            niche = parts[1].strip()
        else:
            niche = text.strip()
        niche = niche.replace("_", " ")
        niche = niche.replace(" stroma", "")
        niche = niche.replace(" spaces", "")
        niche = niche.replace("surface", "surf.")
        niche = niche.replace(" structures", "")
        niche = niche.replace("Smooth", "S. ")
        niche = niche.replace("Desmoplastic", "Desmopl.")
        niche = niche.replace("Macrophage", "Mac.")
        niche = niche.replace("aggregates", "aggr.")
        niche_names.append(niche)
    return niche_names

def make_heatmap_for_celltype(
    df_pivot: pd.DataFrame, 
    df_annot: pd.DataFrame, 
    celltype: str, 
    save: bool = True, 
    ordered_rows = None,
    ordered_cols = None,
):
    tmp = df_pivot.reset_index()
    tmp = tmp.loc[tmp.celltype_combo == celltype].set_index(["cytokine", "celltype_combo"]).T
    tmp = tmp.T.reset_index().set_index("cytokine").drop("celltype_combo", axis=1).T
    tmp.columns = replace_greek(tmp.columns)

    tmp_annot = df_annot.reset_index()
    tmp_annot = tmp_annot.loc[tmp_annot.celltype_combo == celltype].set_index(["cytokine", "celltype_combo"]).T
    tmp_annot = tmp_annot.T.reset_index().set_index("cytokine").drop("celltype_combo", axis=1).T
    tmp_annot.columns = replace_greek(tmp_annot.columns)

    if ordered_cols is None:
        col_linkage = linkage(tmp.T, method='average', metric='euclidean')
        col_dendrogram = dendrogram(col_linkage, no_plot=True)
        ordered_cols = [tmp.columns[i] for i in col_dendrogram['leaves']]
        ordered_cols = sorted(tmp.columns)
    else:
        _columns = replace_greek(tmp.columns)
        ordered_cols = replace_greek(ordered_cols)
        ordered_cols = np.asarray(ordered_cols)[np.isin(ordered_cols, _columns)]
        
    if ordered_rows is None:
        row_linkage = linkage(tmp, method='average', metric='euclidean')
        row_dendrogram = dendrogram(row_linkage, no_plot=True)
        ordered_rows = [tmp.index[i] for i in row_dendrogram['leaves']]
    tmp = tmp.reindex(index=ordered_rows, columns=ordered_cols)
    tmp_annot = tmp_annot.reindex(index=ordered_rows, columns=ordered_cols)
    fig, ax = plt.subplots(figsize=(0.18*tmp.shape[1], 0.18*tmp.shape[0]))
    hm = sns.heatmap(tmp, annot=tmp_annot, fmt="", cmap="RdBu_r", center=0, square=True, annot_kws={"fontsize": fs, "family": "Arial"}, linecolor="white", linewidths=0.5, cbar=False, ax=ax)
    plt.xlabel("")
    plt.ylabel("")
    ytick_positions = np.arange(0.5, 0.5 + tmp.shape[0])
    ytick_labels = tmp.index
    ytick_labels_reduced = extract_niche_name(ytick_labels)
    ax.set_yticks(ytick_positions, ytick_labels_reduced, fontsize=fs, rotation=0, family='Arial')
    ax.tick_params(axis='y', which='major', length=0)
    ax.tick_params(axis='x', which='major', length=0)
    xtick_positions = np.arange(0.5, 0.5 + tmp.shape[1])
    celltype_labels = replace_greek(shorten_celltype_labels(tmp.columns.get_level_values(0)))
    ax.set_xticks(xtick_positions, celltype_labels, fontsize=fs, family='Arial')
    ax.tick_params(axis='y', which='major', length=0)
    ax.set_title(celltype.replace("_", " "), fontsize=6, family="Arial")
    if True:
        plt.savefig(f"./final_figures/heatmap_tumor_{celltype}.svg".replace(" ", "_"), pad_inches=0, bbox_inches="tight", transparent=True)
    plt.show()
    return ordered_rows

In [ ]:
cytokine_groups = pd.read_excel("/Users/soeren.becker/repos/HumanCytokineDict/data/SI_tables.xlsx", sheet_name="6.cytokine_groups")
cytokine_groups = dict(
    zip(
        # cytokine_groups.loc[cytokine_groups.loc[:, "Group ID"] < 5, "Group ID"], 
        # cytokine_groups.loc[cytokine_groups.loc[:, "Group ID"] < 5, "Cytokine Groups by Leiden Clustering"]
        cytokine_groups.loc[:, "Group ID"], 
        cytokine_groups.loc[:, "Cytokine Groups by Leiden Clustering"].str.split(", ")
    )
)
cytokine_groups

cytokine_order = [*sorted(cytokine_groups[k] for k in cytokine_groups.keys())]
final_cytokine_order = []
for c in cytokine_order:
    if isinstance(c, str):
        final_cytokine_order.append(c)
    else:
        final_cytokine_order.extend(c)
final_cytokine_order

In [ ]:
final_cytokine_order = [
    'C5a',
    'CD30L',
    'CD40L',
    'GITRL',
    'IFN-ε',
    'IL-1-α',
    'IL-1-β',
    'IL-2',
    'IL-7',
    'IL-15',
    'IL-12',
    'IFN-α1',
    'IFN-β',
    'IFN-ω',
    'IFN-γ',
    'IL-27',
    
    'IL-17D',
    'CD27L',
    'IL-1Ra',
    'IL-21',
    'IL-26',
    'IL-32-β',
    'IL-33',
    'IL-34',
    'IL-4',
    'IL-13',
    'Leptin',
    'M-CSF',
    'G-CSF',
    'GM-CSF',
    'OSM',
    'IL-17C',
    'IGF-1',
    'IFN-λ1',
    'TNF-α',
    'RANKL',
    'EGF',
    'TSLP',
    'IFN-λ3',
    'BAFF',
    'TL1A',
    'FGF-β',
    'Noggin',
    'IL-3',
    'IL-10',
]

fs = 6

In [ ]:
df_pivot = selected_results.pivot(
    index = ["cytokine", "celltype_combo"],
    columns = "contrast",
    values="NES",
).astype(float)

df_pivot = df_pivot.sort_index()
# df_pivot = df_pivot.loc[:, niche_order]

df_annot = df_pivot.copy()
with warnings.catch_warnings():
    warnings.simplefilter(action='ignore', category=FutureWarning)
    df_annot[:] = ""

for (cytokine, celltype_combo) in df_pivot.index:
    for contrast in df_pivot.columns:
        selected = robust_results.loc[(robust_results.contrast == contrast) & (robust_results.cytokine == cytokine) & (robust_results.celltype_combo == celltype_combo)]
        if selected.shape[0] > 0:
            if selected.qval_threshold.values[0] == 0.1:
                df_annot.loc[(cytokine, celltype_combo), contrast] = "*"
            elif selected.qval_threshold.values[0] == 0.05:
                df_annot.loc[(cytokine, celltype_combo), contrast] = "**"
            elif selected.qval_threshold.values[0] == 0.01:
                df_annot.loc[(cytokine, celltype_combo), contrast] = "***"

In [ ]:
df_pivot.columns

In [ ]:
niche_order = [
    'tumor_vs_Alveolar_spaces',
    'tumor_vs_Smooth_muscle_structures',
    'tumor_vs_Airways',
    'tumor_vs_Desmoplastic_stroma',
    'tumor_vs_Vascular_stroma',
    'tumor_vs_T_cell_aggregates',
    'tumor_vs_Macrophage_islands',   
    'tumor_vs_DC_islands',
]

In [ ]:
make_heatmap_for_celltype(df_pivot, df_annot, celltype = "Monocytes (CD14_Mono)", ordered_rows=niche_order, ordered_cols=final_cytokine_order)
make_heatmap_for_celltype(df_pivot, df_annot, celltype = 'Cytotoxic T cells (CD8_T_cell)', ordered_rows=niche_order, ordered_cols=final_cytokine_order)
make_heatmap_for_celltype(df_pivot, df_annot, celltype = 'Dendritic cells (cDC)', ordered_rows=niche_order, ordered_cols=final_cytokine_order)
make_heatmap_for_celltype(df_pivot, df_annot, celltype = 'Regulatory T cells (Treg)', ordered_rows=niche_order, ordered_cols=final_cytokine_order);

In [ ]:
def custom_cmap(cmap, vmin, vmax, center):
    cmap = plt.get_cmap(cmap)
    vrange = max(vmax - center, center - vmin)
    normlize = mcolors.Normalize(center - vrange, center + vrange)
    cmin, cmax = normlize([vmin, vmax])
    cc = np.linspace(cmin, cmax, 512)
    cmap2 = mcolors.ListedColormap(cmap(cc))
    return cmap2

vmin = df_pivot.min().min()
vmax = df_pivot.max().max()
center = 0
cmap = custom_cmap("RdBu_r", vmin=vmin, vmax=vmax, center=center)
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
fig, ax = plt.subplots(figsize=(0.08, 0.18*8))
cbar = colorbar.ColorbarBase(ax, cmap=cmap, norm=norm, orientation='vertical',)
cbar.ax.tick_params(labelsize=6)
cbar.ax.spines[['right', 'left', 'top', 'bottom']].set_visible(False)
cbar.outline.set_visible(False)
cbar.ax.set_yticks([-1.8, -1, 0, 1, 1.8], ["-1.8", "-1", "0", "1", "1.8"], family="Arial", fontsize=6)
cbar.ax.set_ylabel("Norm. enrichment score\n(Tumor vs. rest)", fontsize=6, family="Arial")

if True:
    plt.savefig("./final_figures/heatmap_tumor_colorbar.svg", bbox_inches="tight", pad_inches=0, transparent=True)
plt.show()